# small_fable — GROUNDING v2: GENERIC blocks + CLUBBING

v1 proved **execution grounding** — a frozen Qwen-1.5B follows *concrete* English plans ("Keep only the gadgets that are waterproof"). But those plans copy the instruction's words, so they can't become a **reusable** primitive vocab. **v2 tests two things v1 couldn't:**

1. **Generic blocks (abstraction grounding):** the criteria move *into the problem* as a named ordered list ("Requirements: 1) waterproof, 2) wireless; Preference: cheapest"), and every plan block uses **only universal-primitive verbs + positional references** — *"Keep only the items that satisfy the 1st stated requirement."* — and **never a domain word**. A hard assertion guarantees 0 instruction-word leaks across all blocks. This is the grounding a reusable vocab actually requires.
2. **Clubbing:** `--club 1` = one operation per block; `--club 2` = **two operations bundled into one block** ("…satisfy the 1st requirement, then from those keep only the 2nd…"). Same problems & answers across club levels (same seed) — so running the probe on each gives a clean **club-1 vs club-2** following comparison. *Can Qwen handle 2 clubbed ops per block?*

Same frozen, judge-free probe (`grounding_test.py`, unchanged). The honest number is the **conditional** `neg_follow` (over rows the model fails unaided); the headline output is the **cost of clubbing** = c2 − c1.

## 0 · GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU — runs on CPU (slow). Colab: Runtime -> GPU.'

## 1 · Clone & install (pulls the latest commit)

In [ ]:
import os
!rm -rf small_fable
!git clone -q https://github.com/sinha-k-prat/small_fable.git
%cd small_fable
!git log --oneline -1
!pip install -q -r requirements.txt
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('\nReady.')

## 2 · Build BOTH club datasets (deterministic, instant)
Same problems/answers; only the block phrasing/clubbing differs. Note: no domain words in any block.

In [ ]:
!python -u tools/gen_grounding_blocks.py --club 1 --out grounding_blocks_c1.jsonl
!python -u tools/gen_grounding_blocks.py --club 2 --out grounding_blocks_c2.jsonl
import json
c1=[json.loads(l) for l in open('grounding_blocks_c1.jsonl')]
c2=[json.loads(l) for l in open('grounding_blocks_c2.jsonl')]
import statistics as st
print(f'rows c1={len(c1)} c2={len(c2)} | mean blocks c1={st.mean(r["n_turns"] for r in c1):.2f} c2={st.mean(r["n_turns"] for r in c2):.2f}')
r=next(x for x in c1 if x['topic']=='constraint_select'); r2=next(x for x in c2 if x['id']==r['id'])
print('\nGENERIC blocks (no instruction words) — same problem, club 1 vs club 2:')
print(' club1:', r['gold_plan'])
print(' club2:', r2['gold_plan'], '   <- 2 ops bundled in block 1')
print(' answer (both):', r['gold_answer'])

## 3 · Run the probe on each club level (the SAME grounding_test.py)
A100 → `bf16`; T4 → `fp16`. Watch `marker_rate ≥ 90%` for each; if c2 is lower (clubbed → longer CoT), bump `--max_new_tokens` to 384 and rerun that one.

In [ ]:
DTYPE='fp16'  # 'bf16' on A100
!python -u grounding_test.py --data grounding_blocks_c1.jsonl --base Qwen/Qwen2.5-1.5B-Instruct \
    --dtype $DTYPE --device cuda --max_new_tokens 320 --out grounding_out_c1
print('\n================ now club=2 ================\n')
!python -u grounding_test.py --data grounding_blocks_c2.jsonl --base Qwen/Qwen2.5-1.5B-Instruct \
    --dtype $DTYPE --device cuda --max_new_tokens 320 --out grounding_out_c2

## 4 · Granular comparison — ALL categories, club=1 (1 primitive/block) vs club=2 (2 clubbed)
`compare_clubbing.py` prints a per-category table: for **every topic**, `neg_follow` / `neg_to_gold` / `acc_plan` / `acc_noplan` for club=1 and club=2 side by side, with the **Δ = club2 − club1**. The bottom rows give OVERALL (all rows) and OVERALL (conditional = only rows the model fails unaided — the honest grounding number). A big negative `neg_follow` Δ (or positive `neg_to_gold` Δ) in a category means bundling two primitives into one block hurt following **there**. Read each category against its `acc_noplan`: the delta only means something where `acc_noplan` is LOW (the plan is actually load-bearing).

In [ ]:
!python compare_clubbing.py

## 5 · The two verdict plots (generic blocks)

In [ ]:
from IPython.display import Image, display
import os
for tag in ('c1','c2'):
    p=f'grounding_out_{tag}/grounding.png'
    print(tag.upper()+':'); display(Image(filename=p)) if os.path.exists(p) else print(' missing',p)